In [1]:
import os
import pandas as pd
import numpy as np


CONTINUOUS_FEATURES = [

    'LapTime_Seconds',
    'Position',
    'LapNumber',
    'Stint',
    'TyreLife',
    'TrackStatus',

    'delta_laptime',
    'CumulativeTimeStint',
    'race_progress_fraction',

    'relative_tire_age',
    'tire_compound_age_delta',
    'tire_performance_decay',
    'rolling_pace_mean_5',
    'pace_std_5',
    'pace_degradation_slope',

    'historical_pit_lap',
    'pit_window_delta',

    'Speed_mean',
    'Speed_max',
    'Speed_std',

    'RPM_mean',
    'RPM_max',

    'Throttle_mean',
    'Throttle_std',

    'Brake_mean',
    'Brake_sum',

    'DRS_mean',
    'DRS_sum',

    'DistanceToDriverAhead_mean',
    'DistanceToDriverAhead_min',

    'nGear_mean',
    'nGear_max',

    'traffic_pressure',
    'drs_dependency',
    'thermal_stress_proxy',
    'high_speed_stress',
    'brake_aggression',
    'pace_vs_ahead'
]


CATEGORICAL_FEATURES = [
    'Compound',
    'Team'
]


METADATA_COLUMNS = [
    'Year',
    'RaceID',
    'Driver',
    'DriverNumber'
]


TARGET_COLUMN = [
    'HasPitStop'
]


def load_processed_files(processed_root):

    all_dfs = []

    for year in os.listdir(processed_root):

        year_path = os.path.join(
            processed_root,
            year
        )

        if not os.path.isdir(year_path):
            continue

        for file in os.listdir(year_path):

            if not file.endswith("_processed.csv"):
                continue

            file_path = os.path.join(
                year_path,
                file
            )

            try:

                df = pd.read_csv(
                    file_path
                )

                race_id = file.replace(
                    "_processed.csv",
                    ""
                )

                df['Year'] = int(year)

                df['RaceID'] = race_id

                all_dfs.append(df)

                print(
                    f"Loaded {year} {race_id} "
                    f"({len(df)} rows)"
                )

            except Exception as e:

                print(
                    f"FAILED {file}: {e}"
                )

    if len(all_dfs) == 0:
        raise ValueError(
            "No processed files found"
        )

    master_df = pd.concat(
        all_dfs,
        ignore_index=True
    )

    return master_df


def select_training_columns(df):

    required_columns = (
        METADATA_COLUMNS +
        CONTINUOUS_FEATURES +
        CATEGORICAL_FEATURES +
        TARGET_COLUMN
    )

    existing_columns = [
        col for col in required_columns
        if col in df.columns
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if len(missing_columns) > 0:

        print("\nMissing Columns:")
        for col in missing_columns:
            print(col)

    df = df[
        existing_columns
    ].copy()

    return df


def clean_training_dataframe(df):

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    df[numeric_cols] = (
        df[numeric_cols]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    df[numeric_cols] = (
        df[numeric_cols]
        .fillna(0)
    )

    categorical_cols = [
        col for col in CATEGORICAL_FEATURES
        if col in df.columns
    ]

    for col in categorical_cols:

        df[col] = (
            df[col]
            .astype(str)
            .fillna("UNKNOWN")
        )

    return df


def save_master_training_csv(
    processed_root,
    output_csv="all_training_data.csv"
):

    print("\nLoading processed race files...")

    df = load_processed_files(
        processed_root
    )

    print("\nSelecting features...")

    df = select_training_columns(
        df
    )

    print("\nCleaning dataframe...")

    df = clean_training_dataframe(
        df
    )

    print("\nDataset Summary")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")

    print("\nYears:")
    print(
        sorted(
            df['Year'].unique()
        )
    )

    print("\nPit Stop Distribution:")
    print(
        df['HasPitStop']
        .value_counts()
    )

    df.to_csv(
        output_csv,
        index=False
    )

    print(
        f"\nSaved training dataset to:\n"
        f"{output_csv}"
    )

    return df


if __name__ == "__main__":

    PROCESSED_ROOT = (
        r"/content/drive/Othercomputers/My Computer/Nguyen Tri/Code/Statisanalyss/Preprocessed"
    )

    OUTPUT_CSV = (
        r"/content/drive/Othercomputers/My Computer/Nguyen Tri/Code/Statisanalyss/all_training_data.csv"
    )

    df = save_master_training_csv(
        processed_root=PROCESSED_ROOT,
        output_csv=OUTPUT_CSV
    )

    print("\nDONE")


Loading processed race files...
Loaded 2018 Abu Dhabi Grand Prix (0 rows)
Loaded 2018 Austrian Grand Prix (791 rows)
Loaded 2018 Azerbaijan Grand Prix (294 rows)
Loaded 2018 Belgian Grand Prix (329 rows)
Loaded 2018 Brazilian Grand Prix (921 rows)
Loaded 2018 British Grand Prix (881 rows)
Loaded 2018 Canadian Grand Prix (0 rows)
Loaded 2018 Chinese Grand Prix (748 rows)
Loaded 2018 French Grand Prix (358 rows)
Loaded 2018 German Grand Prix (749 rows)
Loaded 2018 Hungarian Grand Prix (893 rows)
Loaded 2018 Japanese Grand Prix (765 rows)
Loaded 2018 Mexican Grand Prix (0 rows)
Loaded 2018 Monaco Grand Prix (0 rows)
Loaded 2018 Russian Grand Prix (715 rows)
Loaded 2018 Singapore Grand Prix (397 rows)
Loaded 2018 Spanish Grand Prix (961 rows)
Loaded 2018 United States Grand Prix (501 rows)
Loaded 2019 Abu Dhabi Grand Prix (1075 rows)
Loaded 2019 Australian Grand Prix (1041 rows)
Loaded 2019 Austrian Grand Prix (1401 rows)
Loaded 2019 Azerbaijan Grand Prix (948 rows)
Loaded 2019 Bahrain Gr

/tmp/ipykernel_4046/3371195312.py:136: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat(



Cleaning dataframe...

Dataset Summary
Rows: 152475
Columns: 43

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Pit Stop Distribution:
HasPitStop
0    147657
1      4818
Name: count, dtype: int64

Saved training dataset to:
/content/drive/Othercomputers/My Computer/Nguyen Tri/Code/Statisanalyss/all_training_data.csv

DONE


In [3]:
print(df.head())
df.describe()

   Year               RaceID DriverNumber  LapTime_Seconds  Position  \
0  2018  Austrian Grand Prix           10           87.790      15.0   
1  2018  Austrian Grand Prix           10           69.364      15.0   
2  2018  Austrian Grand Prix           10           69.872      15.0   
3  2018  Austrian Grand Prix           10           70.013      15.0   
4  2018  Austrian Grand Prix           10           69.810      15.0   

   LapNumber  TyreLife  TrackStatus  delta_laptime  CumulativeTimeStint  ...  \
0       16.0       1.0        671.0          0.000               87.790  ...   
1       17.0       2.0          1.0        -18.426              157.154  ...   
2       18.0       3.0          1.0          0.508              227.026  ...   
3       19.0       4.0          1.0          0.141              297.039  ...   
4       20.0       5.0          1.0         -0.203              366.849  ...   

   nGear_max  traffic_pressure  drs_dependency  thermal_stress_proxy  \
0        7.0  

,Year,LapTime_Seconds,Position,LapNumber,TyreLife,TrackStatus,delta_laptime,CumulativeTimeStint,race_progress_fraction,relative_tire_age,...,DistanceToDriverAhead_mean,DistanceToDriverAhead_min,nGear_mean,nGear_max,traffic_pressure,drs_dependency,thermal_stress_proxy,high_speed_stress,brake_aggression,pace_vs_ahead
count,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,...,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,1.524750e+05,152475.000000,152475.000000,152475.000000
mean,2021.856954,89.645814,9.800846,31.051471,15.143965,15.540856,-0.299531,1280.514121,0.504180,0.518473,...,296.180100,129.524359,5.061855,7.831133,0.025133,5.603816,6.009870e+05,12556.644479,27989.261601,4.988154
std,2.215135,30.115262,5.451492,18.222725,10.641102,586.532276,19.513392,924.712744,0.284748,0.383811,...,768.133399,675.855369,0.749966,0.995827,0.080951,112.713917,3.354821e+05,3139.667581,8553.780075,16.508729
min,2018.000000,0.000000,0.000000,1.000000,0.000000,1.000000,-2414.122000,0.000000,0.011494,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,2020.000000,80.454500,5.000000,16.000000,7.000000,1.000000,-0.303000,549.237500,0.258621,0.225000,...,66.310091,0.180000,4.803112,8.000000,0.003205,0.000000,4.852164e+05,11077.925480,23986.641000,0.630414
50%,2022.000000,88.233000,10.000000,30.000000,13.000000,1.000000,0.000000,1108.624000,0.500000,0.444444,...,141.865182,0.884444,5.165498,8.000000,0.006964,0.000000,5.744897e+05,12870.489515,27324.338557,1.374154
75%,2024.000000,99.109500,14.000000,45.000000,21.000000,1.000000,0.280000,1814.943500,0.750000,0.722222,...,308.840892,46.716944,5.527589,8.000000,0.014765,4.046596,6.861277e+05,14586.257120,31233.471996,2.877515
max,2025.000000,2526.253000,20.000000,87.000000,78.000000,67124.000000,71.028000,6158.100000,1.000000,3.111111,...,149179.430030,147483.182779,39.207407,128.000000,0.987253,8651.586424,5.628041e+07,27324.545161,817631.997964,225.269432


In [4]:
!pip install tensorflow

In [5]:
import tensorflow as tf

from keras import Model
from keras.layers import (
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)

class F1PitStopPredictor(Model):
    def __init__(self):
        super(F1PitStopPredictor, self).__init__()

        # =========================
        # Bi-LSTM Layer 1
        # =========================
        self.bilstm1 = Bidirectional(
            LSTM(
                256,
                return_sequences=True,
                recurrent_dropout=0.2
            )
        )

        self.dropout1 = Dropout(0.2)
        self.bn1 = BatchNormalization()

        # =========================
        # Bi-LSTM Layer 2
        # =========================
        self.bilstm2 = Bidirectional(
            LSTM(
                128,
                return_sequences=True,
                recurrent_dropout=0.2
            )
        )

        self.dropout2 = Dropout(0.3)
        self.bn2 = BatchNormalization()

        # =========================
        # Bi-LSTM Layer 3
        # =========================
        self.bilstm3 = Bidirectional(
            LSTM(
                64,
                return_sequences=False,
                recurrent_dropout=0.2
            )
        )

        self.dropout3 = Dropout(0.3)
        self.bn3 = BatchNormalization()

        # =========================
        # Dense Layers
        # =========================
        self.dense1 = Dense(64, activation='relu')
        self.dropout4 = Dropout(0.2)

        self.dense2 = Dense(32, activation='relu')

        # Final Output
        self.output_layer = Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):

        x = self.bilstm1(inputs)
        x = self.dropout1(x, training=training)
        x = self.bn1(x, training=training)

        x = self.bilstm2(x)
        x = self.dropout2(x, training=training)
        x = self.bn2(x, training=training)

        x = self.bilstm3(x)
        x = self.dropout3(x, training=training)
        x = self.bn3(x, training=training)

        x = self.dense1(x)
        x = self.dropout4(x, training=training)

        x = self.dense2(x)

        return self.output_layer(x)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_curve
)

from sklearn.utils.class_weight import compute_class_weight


# =========================================================
# LOAD DATA
# =========================================================

path = "/content/drive/Othercomputers/My Computer/Nguyen Tri/Code/Statisanalyss/all_training_data.csv"

df = pd.read_csv(path)


# =========================================================
# REMOVE LEAKAGE / UNUSED FEATURES
# =========================================================

drop_cols = [

    'PitInTime',
    'PitOutTime',

    'LapTime',
    'Sector1Time',
    'Sector2Time',
    'Sector3Time',

    'SessionTime',
    'Time',

    'Deleted',
    'DeletedReason',

    'FastF1Generated',
    'IsAccurate'
]

existing_drop_cols = [
    col for col in drop_cols
    if col in df.columns
]

df = df.drop(
    columns=existing_drop_cols,
    errors='ignore'
)


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

if 2025 in df['Year'].values:

    train_df = df[
        df['Year'] < 2025
    ].copy()

    test_df = df[
        df['Year'] == 2025
    ].copy()

else:

    latest_year = df['Year'].max()

    print(
        f"Warning: 2025 not found. "
        f"Using {latest_year}"
    )

    train_df = df[
        df['Year'] < latest_year
    ].copy()

    test_df = df[
        df['Year'] == latest_year
    ].copy()


# =========================================================
# FIX HISTORICAL PIT LEAKAGE
# =========================================================

historical_pit = (
    train_df[
        train_df['HasPitStop'] == 1
    ]
    .groupby(
        ['Compound', 'Team']
    )['LapNumber']
    .mean()
    .reset_index()
)

historical_pit.columns = [
    'Compound',
    'Team',
    'historical_pit_lap'
]

train_df = train_df.drop(
    columns=['historical_pit_lap'],
    errors='ignore'
)

test_df = test_df.drop(
    columns=['historical_pit_lap'],
    errors='ignore'
)

train_df = train_df.merge(
    historical_pit,
    on=['Compound', 'Team'],
    how='left'
)

test_df = test_df.merge(
    historical_pit,
    on=['Compound', 'Team'],
    how='left'
)

global_pit_mean = (
    historical_pit[
        'historical_pit_lap'
    ].mean()
)

train_df['historical_pit_lap'] = (
    train_df['historical_pit_lap']
    .fillna(global_pit_mean)
)

test_df['historical_pit_lap'] = (
    test_df['historical_pit_lap']
    .fillna(global_pit_mean)
)

train_df['pit_window_delta'] = (
    train_df['LapNumber'] -
    train_df['historical_pit_lap']
)

test_df['pit_window_delta'] = (
    test_df['LapNumber'] -
    test_df['historical_pit_lap']
)


# =========================================================
# FEATURES
# =========================================================

categorical_cols = [
    'Compound',
    'Team'
]

continuous_cols = [

    'LapTime_Seconds',
    'Position',
    'LapNumber',
    'TyreLife',
    'TrackStatus',

    'delta_laptime',
    'CumulativeTimeStint',
    'race_progress_fraction',

    'relative_tire_age',
    'tire_compound_age_delta',
    'tire_performance_decay',
    'rolling_pace_mean_5',
    'pace_std_5',
    'pace_degradation_slope',

    'historical_pit_lap',
    'pit_window_delta',

    'Speed_mean',
    'Speed_max',
    'Speed_std',

    'RPM_mean',
    'RPM_max',

    'Throttle_mean',
    'Throttle_std',

    'Brake_mean',
    'Brake_sum',

    'DRS_mean',
    'DRS_sum',

    'DistanceToDriverAhead_mean',
    'DistanceToDriverAhead_min',

    'nGear_mean',
    'nGear_max',

    'traffic_pressure',
    'drs_dependency',
    'thermal_stress_proxy',
    'high_speed_stress',
    'brake_aggression',
    'pace_vs_ahead'
]


# =========================================================
# ENCODE CATEGORICAL FEATURES
# =========================================================

combined_df = pd.concat(
    [train_df, test_df],
    axis=0
)

combined_df = pd.get_dummies(
    combined_df,
    columns=categorical_cols,
    drop_first=False
)

train_df = combined_df[
    combined_df['Year'] < 2025
].copy()

test_df = combined_df[
    combined_df['Year'] == 2025
].copy()

encoded_cat_cols = [

    col for col in combined_df.columns

    if (
        col.startswith("Compound_")
        or col.startswith("Team_")
    )
]

feature_cols = (
    continuous_cols +
    encoded_cat_cols
)


# =========================================================
# SCALE FEATURES
# =========================================================

scaler = StandardScaler()

train_df[continuous_cols] = (
    scaler.fit_transform(
        train_df[continuous_cols]
    )
)

test_df[continuous_cols] = (
    scaler.transform(
        test_df[continuous_cols]
    )
)

train_df = train_df.fillna(0)
test_df = test_df.fillna(0)


# =========================================================
# SEQUENCE CREATION
# =========================================================

def create_driver_sequences(
    data,
    feature_columns,
    target_column='HasPitStop',
    seq_length=20
):

    X = []
    y = []

    grouped = data.groupby(
        ['RaceID', 'DriverNumber']
    )

    for _, group in grouped:

        group = group.sort_values(
            'LapNumber'
        )

        feat = group[
            feature_columns
        ].values

        targ = group[
            target_column
        ].values

        if len(feat) < seq_length:
            continue

        for i in range(
            len(feat) - seq_length + 1
        ):

            X.append(
                feat[
                    i:i+seq_length
                ]
            )

            y.append(
                targ[
                    i+seq_length-1
                ]
            )

    return (
        np.array(
            X,
            dtype=np.float32
        ),
        np.array(
            y,
            dtype=np.float32
        )
    )


X_train, y_train = create_driver_sequences(
    train_df,
    feature_cols
)

X_test, y_test = create_driver_sequences(
    test_df,
    feature_cols
)

print(X_train.shape)
print(X_test.shape)


# =========================================================
# CLASS WEIGHTS
# =========================================================

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)


# =========================================================
# FOCAL LOSS
# =========================================================

def focal_loss(
    gamma=2.0,
    alpha=0.25
):

    def loss(
        y_true,
        y_pred
    ):

        y_true = tf.cast(
            y_true,
            tf.float32
        )

        bce = (
            tf.keras.backend.binary_crossentropy(
                y_true,
                y_pred
            )
        )

        p_t = (
            y_true * y_pred
            +
            (1 - y_true)
            *
            (1 - y_pred)
        )

        alpha_factor = (
            y_true * alpha
            +
            (1 - y_true)
            *
            (1 - alpha)
        )

        modulating_factor = tf.pow(
            (1 - p_t),
            gamma
        )

        return tf.reduce_mean(
            alpha_factor
            *
            modulating_factor
            *
            bce
        )

    return loss


# =========================================================
# MODEL
# =========================================================

pit_stop_model = F1PitStopPredictor()


# =========================================================
# COMPILE
# =========================================================

pit_stop_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss=focal_loss(),

    metrics=[

        'accuracy',

        tf.keras.metrics.AUC(
            curve='ROC',
            name='roc_auc'
        ),

        tf.keras.metrics.AUC(
            curve='PR',
            name='pr_auc'
        )
    ]
)


# =========================================================
# CALLBACKS
# =========================================================

early_stopping = tf.keras.callbacks.EarlyStopping(

    monitor='val_pr_auc',

    mode='max',

    patience=8,

    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(

    monitor='val_pr_auc',

    mode='max',

    factor=0.5,

    patience=4,

    min_lr=1e-6
)


# =========================================================
# TRAIN
# =========================================================

history = pit_stop_model.fit(

    X_train,
    y_train,

    validation_data=(
        X_test,
        y_test
    ),

    batch_size=64,

    epochs=50,

    class_weight=class_weights,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)


# =========================================================
# PREDICT
# =========================================================

y_pred_probs = (
    pit_stop_model
    .predict(
        X_test,
        batch_size=256
    )
    .flatten()
)


# =========================================================
# BEST THRESHOLD
# =========================================================

precision_vals, recall_vals, thresholds = (
    precision_recall_curve(
        y_test,
        y_pred_probs
    )
)

f1_scores = (
    2
    *
    precision_vals
    *
    recall_vals
    /
    (
        precision_vals
        +
        recall_vals
        +
        1e-8
    )
)

best_idx = np.argmax(
    f1_scores[:-1]
)

best_threshold = thresholds[
    best_idx
]

print(
    f"Best Threshold: "
    f"{best_threshold:.4f}"
)


# =========================================================
# FINAL PREDICTIONS
# =========================================================

y_pred = (
    y_pred_probs >= best_threshold
).astype(int)


# =========================================================
# METRICS
# =========================================================

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_pred_probs
)

auc_pr = average_precision_score(
    y_test,
    y_pred_probs
)

balanced_acc = balanced_accuracy_score(
    y_test,
    y_pred
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()

specificity = tn / (tn + fp)


# =========================================================
# RESULTS
# =========================================================

print("\n--- Evaluation Metrics ---")

print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1-Score:          {f1:.4f}")
print(f"Specificity:       {specificity:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"AUC-PR:            {auc_pr:.4f}")

print("\n--- Confusion Matrix ---")

print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")


# =========================================================
# SUMMARY
# =========================================================

pit_stop_model.build(
    input_shape=(
        None,
        X_train.shape[1],
        X_train.shape[2]
    )
)

pit_stop_model.summary()


# =========================================================
# SAVE MODEL
# =========================================================

pit_stop_model.save(
    "f1_bilstm.keras"
)

print("\nModel Saved")

(111954, 20, 59)
(14833, 20, 59)
{0: np.float64(0.5163644078741029), 1: np.float64(15.777057497181511)}
Epoch 1/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 817s 458ms/step - accuracy: 0.9560 - loss: 0.0226 - pr_auc: 0.0406 - roc_auc: 0.5974 - val_accuracy: 0.9697 - val_loss: 0.0123 - val_pr_auc: 0.0801 - val_roc_auc: 0.7362 - learning_rate: 1.0000e-04
Epoch 2/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 824s 436ms/step - accuracy: 0.9683 - loss: 0.0146 - pr_auc: 0.1131 - roc_auc: 0.7709 - val_accuracy: 0.9697 - val_loss: 0.0093 - val_pr_auc: 0.2840 - val_roc_auc: 0.9048 - learning_rate: 1.0000e-04
Epoch 3/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 784s 448ms/step - accuracy: 0.9685 - loss: 0.0111 - pr_auc: 0.3036 - roc_auc: 0.8953 - val_accuracy: 0.9703 - val_loss: 0.0069 - val_pr_auc: 0.6623 - val_roc_auc: 0.9637 - learning_rate: 1.0000e-04
Epoch 4/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 760s 435ms/step - accuracy: 0.9706 - loss: 0.0083 - pr_auc: 0.5394 - roc_auc: 0.9540 - val_accuracy: 0.9827 - val_loss: 0.0048 - val_p